# Lab: structured outputs for clause classification

**Session 4: Advanced prompting and structured outputs**

A law firm wants every incoming contract clause tagged with one of 8 labels (termination, payment, other...). You build the classifier with a remote LLM in four versions, changing one thing at a time, and compare their accuracy, their errors and their cost:

1. **V1:** ask for the label in free text, then parse the text yourself
2. **V2:** structured output: the model must answer with a Pydantic schema
3. **V3:** V2 + 5 similar examples in the prompt (few-shot)
4. **V4:** V3 asked 5 times, majority vote (self-consistency)

You complete **four cells marked TODO**. Each one is followed by a check that prints `OK`. Every other cell is given: read it, then run it.


## Setup

1. Create a file named `.env` **next to this notebook**, containing one line: `OPENROUTER_API_KEY=sk-or-...` (your key). Never share it or commit it.
2. Install the packages: `pip install openai python-dotenv pydantic scikit-learn`

The lab makes about 200 short API calls. The results table shows what each version cost.


In [ ]:
# %pip install -q openai python-dotenv pydantic scikit-learn

In [ ]:
import os
import time
from collections import Counter, namedtuple
from concurrent.futures import ThreadPoolExecutor
from statistics import mean, median
from typing import Literal

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv(find_dotenv(usecwd=True))  # reads the .env file next to this notebook
assert os.getenv(
    "OPENROUTER_API_KEY"
), "No OPENROUTER_API_KEY found: create the .env file (see Setup)"

MODEL = os.getenv("OPENROUTER_MODEL", "~deepseek/deepseek-v4-flash-latest")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    timeout=60,
)
print("model:", MODEL)

## Calling the model (given)

Two functions. Both return the answer and a `Usage`: tokens, cost in USD (as reported by OpenRouter) and seconds.

- `ask(system, user)` returns **free text**
- `ask_structured(system, user, schema)` returns an **instance of the Pydantic class `schema`**. The JSON schema of the class is sent with the request, the model's answer must follow it, and Pydantic validates it.

`NO_REASONING` asks the model to answer directly, without hidden reasoning tokens: a label does not need them, and they would make every call slower and costlier.

A call that fails (network, invalid answer) returns `None` instead of crashing: it counts as an error in the results table.


In [ ]:
Usage = namedtuple("Usage", ["tokens", "cost", "seconds"])
NO_REASONING = {
    "reasoning": {"enabled": False}
}  # answer directly, no hidden "thinking" tokens


def _messages(system, user):
    return ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": user}
    ]


def _usage(response, start):
    return Usage(
        response.usage.total_tokens,
        float(getattr(response.usage, "cost", None) or 0.0),
        time.perf_counter() - start,
    )


def ask(system, user, temperature=0):
    start = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=_messages(system, user),
            temperature=temperature,
            max_tokens=300,
            extra_body=NO_REASONING,
        )
    except Exception as error:
        print("API call failed:", str(error)[:120])
        return None, Usage(0, 0.0, time.perf_counter() - start)
    return response.choices[0].message.content or "", _usage(response, start)


def ask_structured(system, user, schema, temperature=0):
    start = time.perf_counter()
    try:
        response = client.chat.completions.parse(
            model=MODEL,
            messages=_messages(system, user),
            temperature=temperature,
            max_tokens=300,
            response_format=schema,
            extra_body=NO_REASONING,
        )
    except Exception as error:  # API error, or an answer that does not match the schema
        print("API call failed:", str(error)[:120])
        return None, Usage(0, 0.0, time.perf_counter() - start)
    return response.choices[0].message.parsed, _usage(response, start)


text, usage = ask(None, "Say hello in one word.")
print(text, usage)

## The clauses (given)

48 hand-written clauses, 8 labels. Per label, 3 clauses go to the **training pool** (used only to pick few-shot examples in V3) and 3 to the **held-out evaluation set** (used only to measure). Never mix them: taking examples from the evaluation set would inflate the score.


In [ ]:
LABELS = [
    "termination",
    "confidentiality",
    "governing_law",
    "indemnification",
    "limitation_of_liability",
    "payment",
    "intellectual_property",
    "other",
]

CLAUSES = {
    "termination": [
        "Either party may terminate this Agreement upon thirty (30) days' prior written notice to the other party.",
        "This Agreement shall automatically expire on December 31, 2027 unless renewed in writing by both parties.",
        "The Supplier may terminate this Agreement immediately if the Customer fails to cure a material breach within fifteen (15) days of notice.",
        "Upon termination for any reason, the Licensee shall cease all use of the Software and return all copies within ten (10) business days.",
        "Either party may end this Agreement with immediate effect if the other party becomes insolvent or enters into liquidation.",
        "The initial term is two (2) years, after which either party may terminate this Agreement on ninety (90) days' notice.",
    ],
    "confidentiality": [
        "Each party shall keep the other party's Confidential Information strictly confidential and shall not disclose it to any third party.",
        "The Recipient shall use the Confidential Information solely for the purpose of evaluating the proposed transaction.",
        "The obligations of confidentiality shall survive for five (5) years following the disclosure of the relevant information.",
        "Confidential Information does not include information that is or becomes publicly available through no fault of the Recipient.",
        "The Consultant's employees shall access the Client's trade secrets only on a need-to-know basis and under written non-disclosure undertakings.",
        "Upon request, the Recipient shall promptly destroy all documents containing the Discloser's proprietary information.",
    ],
    "governing_law": [
        "This Agreement shall be governed by and construed in accordance with the laws of the State of New York.",
        "Any dispute arising out of this Agreement shall be submitted to the exclusive jurisdiction of the courts of Paris, France.",
        "This contract is subject to English law, without regard to its conflict of laws principles.",
        "All disputes shall be finally settled by arbitration under the ICC Rules, with the seat of arbitration in Geneva.",
        "This Agreement and any non-contractual obligations arising from it are governed by the laws of Germany.",
        "The courts of Delaware shall have sole jurisdiction over any claim relating to this Agreement.",
    ],
    "indemnification": [
        "The Supplier shall indemnify and hold harmless the Customer against all claims, losses and expenses arising from the Supplier's negligence.",
        "The Licensee agrees to defend the Licensor against any third-party claim resulting from the Licensee's misuse of the Software.",
        "Each party shall compensate the other for any losses caused by a breach of its representations and warranties.",
        "The Contractor shall reimburse the Owner for any fines or penalties imposed due to the Contractor's failure to comply with applicable laws.",
        "The Seller shall hold the Buyer harmless from any product liability claims relating to goods delivered under this Agreement.",
        "The indemnifying party shall control the defence and settlement of any claim for which indemnification is sought.",
    ],
    "limitation_of_liability": [
        "In no event shall either party be liable for any indirect, incidental or consequential damages, including loss of profits.",
        "The total aggregate liability of the Provider shall not exceed the fees paid by the Customer in the twelve (12) months preceding the claim.",
        "Nothing in this Agreement limits either party's liability for death or personal injury caused by its negligence or for fraud.",
        "The Company shall not be responsible for any loss of data, except to the extent caused by its gross negligence.",
        "The Supplier's maximum liability for any claim shall be capped at one hundred thousand euros (EUR 100,000).",
        "The Software is provided 'as is' and the Licensor disclaims all liability for its performance, except as expressly stated herein.",
    ],
    "payment": [
        "The Customer shall pay all invoices within thirty (30) days of the invoice date by bank transfer.",
        "Late payments shall bear interest at the rate of one percent (1%) per month until paid in full.",
        "The annual subscription fee of EUR 24,000 is payable in advance on the first day of each contract year.",
        "All amounts payable under this Agreement are exclusive of VAT, which shall be added at the applicable rate.",
        "The Client shall reimburse reasonable travel expenses incurred by the Consultant, subject to prior approval.",
        "The Supplier may increase its prices once per year by giving the Customer sixty (60) days' notice.",
    ],
    "intellectual_property": [
        "All intellectual property rights in the deliverables shall vest in the Client upon full payment of the fees.",
        "The Licensor grants the Licensee a non-exclusive, non-transferable licence to use the Software for its internal business purposes.",
        "Nothing in this Agreement transfers ownership of either party's pre-existing patents, trademarks or copyrights.",
        "The Employee hereby assigns to the Company all inventions conceived during the term of employment.",
        "The Customer shall not reverse engineer, decompile or disassemble the Software.",
        "Each party may use the other party's logo solely to identify it as a customer in marketing materials.",
    ],
    "other": [
        "All notices under this Agreement shall be in writing and delivered to the addresses set out above.",
        "Neither party shall be liable for delays caused by events beyond its reasonable control, including floods, wars and pandemics.",
        "If any provision of this Agreement is held invalid, the remaining provisions shall remain in full force and effect.",
        "This Agreement constitutes the entire agreement between the parties and supersedes all prior negotiations.",
        "Neither party may assign this Agreement without the prior written consent of the other party.",
        "This Agreement may be executed in counterparts, each of which shall be deemed an original.",
    ],
}

train_pool = [(text, label) for label, texts in CLAUSES.items() for text in texts[:3]]
eval_set = [(text, label) for label, texts in CLAUSES.items() for text in texts[3:]]
print(f"{len(train_pool)} training clauses, {len(eval_set)} held-out clauses")

## TODO 1: the output contract

Every version must produce a `ClauseLabel`. Complete the class: its only field, `label`, must accept the 8 labels and **nothing else**.

Hint: `Literal["a", "b"]` accepts only `"a"` or `"b"`; `Literal[tuple(LABELS)]` does the same with the list.


In [ ]:
class ClauseLabel(BaseModel):
    label: str  # TODO: replace str by a type that accepts only the 8 strings of LABELS


assert ClauseLabel(label="payment").label == "payment"
try:
    ClauseLabel(label="totally_harmless")
    raise AssertionError("ClauseLabel must refuse a label outside LABELS")
except ValidationError:
    pass
print("ClauseLabel: OK")
print(
    "JSON schema sent to the model:",
    ClauseLabel.model_json_schema()["properties"]["label"],
)

## Measuring a version (given)

`evaluate` runs one version on the 24 held-out clauses (8 at a time) and adds a row to the results table:

- **accuracy:** share of clauses with the right label
- **errors:** clauses with no valid label (unparseable answer or failed call)
- **tokens:** per clause; **USD / 1k:** cost of classifying 1,000 clauses; **latency:** median seconds per clause


In [ ]:
results = []


def evaluate(name, classify, dataset=eval_set):
    with ThreadPoolExecutor(max_workers=8) as pool:
        outputs = list(pool.map(lambda item: classify(item[0]), dataset))
    rows = []
    for out, (_, gold) in zip(outputs, dataset):
        usages = out["usages"]
        rows.append(
            {
                "correct": out["result"] is not None and out["result"].label == gold,
                "error": out["result"] is None,
                "tokens": sum(u.tokens for u in usages),
                "cost": sum(u.cost for u in usages),
                "seconds": sum(u.seconds for u in usages),
            }
        )
    results.append(
        {
            "version": name,
            "accuracy": mean(r["correct"] for r in rows),
            "errors": sum(r["error"] for r in rows),
            "tokens": mean(r["tokens"] for r in rows),
            "USD / 1k": 1000 * mean(r["cost"] for r in rows),
            "latency": median(r["seconds"] for r in rows),
        }
    )
    show()


def show():
    print(
        f"{'version':<18}{'accuracy':>9}{'errors':>8}{'tokens':>8}{'USD / 1k':>10}{'latency':>9}"
    )
    for r in results:
        print(
            f"{r['version']:<18}{r['accuracy']:>9.0%}{r['errors']:>8}{r['tokens']:>8.0f}"
            f"{r['USD / 1k']:>10.3f}{r['latency']:>8.1f}s"
        )

## V1: free text, parsed by hand (TODO 2)

The naive version: ask for the label, get text back, and parse it. Complete `parse_v1`: clean the text (spaces, quotes, a final dot, upper case), then try to build a `ClauseLabel`. If Pydantic refuses it, return `None`: it counts as an error.


In [ ]:
def parse_v1(text):
    # TODO: clean `text`, then return ClauseLabel(label=...), or None if Pydantic refuses it
    ...


assert parse_v1(" Termination.") == ClauseLabel(
    label="termination"
), "clean spaces, the final dot and upper case"
assert parse_v1('"payment"') == ClauseLabel(label="payment"), "remove the quotes"
assert (
    parse_v1("This clause is about payment.") is None
), "a sentence is not a label: return None"
print("parse_v1: OK")

In [ ]:
V1_PROMPT = (
    "Classify this contract clause into one of: {labels}. "
    "Answer with the label only.\n\nClause: {clause}"
)


def classify_v1(clause):
    text, usage = ask(None, V1_PROMPT.format(labels=", ".join(LABELS), clause=clause))
    result = parse_v1(text) if text is not None else None
    return {"result": result, "usages": [usage], "raw": text}


print(classify_v1(eval_set[0][0]))
evaluate("V1 free text", classify_v1)

## V2: structured output (TODO 3)

Now the model gets a system prompt with the definition of each label, and must answer with the `ClauseLabel` schema. Complete `classify_v2`: call `ask_structured` with `SYSTEM_PROMPT`, the user message and `ClauseLabel`, and return the result in the same format as `classify_v1`.

Keep the arguments `examples` and `temperature`: V3 and V4 reuse this function.


In [ ]:
SYSTEM_PROMPT = """You are a contract analyst. Classify ONE contract clause into exactly one label:
- termination: how or when the agreement ends (notice, breach, expiry, insolvency, effects of termination)
- confidentiality: non-disclosure and protection of confidential information
- governing_law: which law, courts or arbitration apply to disputes
- indemnification: one party compensates or defends the other against claims or losses
- limitation_of_liability: caps, exclusions or disclaimers of a party's liability
- payment: fees, invoices, prices, taxes, interest, expenses
- intellectual_property: ownership, licences or protection of IP and software
- other: anything else (notices, assignment, force majeure, severability, entire agreement...)"""


def user_message(clause, examples=()):
    shots = "".join(f"Clause: {text}\nLabel: {label}\n\n" for text, label in examples)
    return f"{shots}Clause: {clause}"


def classify_v2(clause, examples=(), temperature=0):
    # TODO: call ask_structured(...) with the system prompt, user_message(clause, examples),
    #       ClauseLabel and temperature, then return {"result": ..., "usages": [...]}
    ...


out = classify_v2(eval_set[0][0])
assert out is not None and isinstance(
    out.get("result"), ClauseLabel
), "classify_v2 must return {'result': <the ClauseLabel from ask_structured>, 'usages': [...]}"
print("classify_v2: OK", out)
evaluate("V2 structured", classify_v2)

## V3: few-shot examples (given)

`pick_examples` finds the 5 clauses of the **training pool** most similar to the input (TF-IDF cosine similarity; a real system would use embeddings) and puts them in the prompt, with their labels, before the clause to classify.


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2)).fit([text for text, _ in train_pool])
pool_vectors = vectorizer.transform([text for text, _ in train_pool])


def pick_examples(clause, k=5):
    similarities = cosine_similarity(vectorizer.transform([clause]), pool_vectors)[0]
    return [train_pool[i] for i in similarities.argsort()[::-1][:k]]


def classify_v3(clause):
    return classify_v2(clause, examples=pick_examples(clause))


print(user_message(eval_set[0][0], pick_examples(eval_set[0][0], k=2)))
evaluate("V3 few-shot", classify_v3)

## V4: self-consistency (TODO 4)

Ask the V3 question 5 times at temperature 0.7, so the answers can differ, and keep the label that comes up most often. The share of votes it gets (the **agreement**) says how sure the model is.

Complete `majority`: it receives a list of labels and returns the most common one and its share of the votes.


In [ ]:
def majority(labels):
    # TODO: return (the most common label, its share of the votes); hint: collections.Counter
    ...


assert majority(["payment", "other", "payment"]) == (
    "payment",
    2 / 3,
), "expected ('payment', 2/3)"
assert majority(["other"] * 5) == ("other", 1.0), "expected ('other', 1.0)"
print("majority: OK")

In [ ]:
def classify_v4(clause, k=5):
    examples = pick_examples(clause)
    runs = [classify_v2(clause, examples, temperature=0.7) for _ in range(k)]
    usages = [u for run in runs for u in run["usages"]]
    labels = [run["result"].label for run in runs if run["result"] is not None]
    if not labels:
        return {"result": None, "usages": usages}
    label, agreement = majority(labels)
    return {
        "result": ClauseLabel(label=label),
        "agreement": agreement,
        "usages": usages,
    }


print(classify_v4(eval_set[0][0]))
evaluate("V4 self-consist.", classify_v4)

## Read your table

1. How many errors does V1 make, and V2? What did the schema change?
2. Which version has the best accuracy? Which one has the best accuracy **per dollar**?
3. With 24 held-out clauses, one clause is about 4 points of accuracy. Is the gap between two versions bigger than that?
4. Run the V4 cell again: do the results change? Why?


In [ ]:
spent = sum(r["USD / 1k"] / 1000 * len(eval_set) for r in results)
print(f"This lab cost about USD {spent:.4f} for the {len(results)} rows of the table.")